With a strong El Niño under way and likely to continue through the winter (a greater than 79% chance,
according to [NOAA's ENSO discussion of 12 November 2026](https://www.cpc.ncep.noaa.gov/products/analysis_monitoring/enso_advisory/ensodisc.shtml)),
it is a good time to look at how ENSO affects agriculture in the Pacific Northwest.

Farms in central Washington get very little rain in summer. The water for orchards, hop yards and vineyards
comes from snow that falls in the Cascades in winter and melts from spring into summer. So a question
for anyone planning an irrigation season is: **does the winter ENSO phase (El Niño or La Niña) tell us
anything about how much water the rivers will carry from April to September?**

We compare two rivers that supply irrigated farmland:

- the **Yakima River**, which supplies the Yakima Valley (apples, cherries, hops, wine grapes). Its reservoirs
  are small compared with the river's yearly flow, so a poor snow year shows up directly in summer water.
- the **Columbia River**, which supplies the Columbia Basin. It drains a huge area in Canada and several US
  states and is controlled by many large dams.

Data: daily streamflow from USGS stream gauges and NOAA's Relative Oceanic Niño Index (RONI), both through
the project's data sources (`capstone.sources`). No login is needed.

In [60]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

from capstone.sources import roni, streamflow

pio.renderers.default = "notebook_connected"  # interactive charts that also work on the website

GAUGES = {
    "USGS-12484500": "Yakima at Umtanum",
    "USGS-12510500": "Yakima at Kiona",
    "USGS-12472800": "Columbia below Priest Rapids Dam",
}
FIRST_WY, LAST_WY = 1950, 2026  # water years: October to September, named after the year they end in

Note that we are using the water years 1950-2026 for this analysis and water years are designated by the calendar year in which they end.

## The gauges

- **Yakima River at Umtanum**: above the large irrigation diversions near Yakima, so mostly the water coming
  down from the mountains and reservoirs.
- **Yakima River at Kiona**: near the mouth, what is left after the valley's irrigation.
- **Columbia River below Priest Rapids Dam**: the Columbia after the big dams of central Washington.

In [61]:
gauges = streamflow.gauges(gauge_ids=list(GAUGES))
gauges["river"] = gauges["gauge_id"].map(GAUGES)
gauges

,gauge_id,name,county,lat,lon,drainage_area_sq_mi,river
0,USGS-12472800,"COLUMBIA RIVER BELOW PRIEST RAPIDS DAM, WA",Grant County,46.628708,-119.864736,96000.0,Columbia below Priest Rapids Dam
1,USGS-12484500,"YAKIMA RIVER AT UMTANUM, WA",Kittitas County,46.862626,-120.480067,1594.0,Yakima at Umtanum
2,USGS-12510500,"YAKIMA RIVER AT KIONA, WA",Benton County,46.253467,-119.478079,5615.0,Yakima at Kiona


Below is a map showing the locations of the gauges used:

In [62]:
import folium

m = folium.Map(location=[46.8, -120.2], zoom_start=8, tiles="OpenStreetMap")
for g in gauges.itertuples():
    folium.Marker(
        [g.lat, g.lon],
        tooltip=f"{g.river} ({g.gauge_id})",
        popup=f"{g.name}<br>Drainage area: {g.drainage_area_sq_mi:,.0f} sq mi",
    ).add_to(m)
m

## El Niño and La Niña

RONI measures how much warmer or colder than normal the tropical Pacific is. We use the value for
December to February (DJF), the heart of the snow season, and call a winter **El Niño** at +0.5 °C or more
and **La Niña** at −0.5 °C or less. In the Pacific Northwest, La Niña winters tend to be cooler and snowier,
and El Niño winters warmer and drier.

In [63]:
roni.chart()

In [64]:
enso = roni.load()
enso = enso[enso["season"] == "DJF"].set_index("year")["anomaly"].rename("roni_djf")
phase = pd.cut(enso, [-10, -0.5, 0.5, 10], labels=["La Niña", "Neutral", "El Niño"]).rename("phase")

Each water year gets the phase of its own winter: water year 2015 (October 2014 to September 2015) is
classified by December 2014 to February 2015, the winter just before its irrigation season.

## Streamflow, 1950 to 2026

This downloads about 80,000 daily values from the USGS Water Data API (around 20 seconds).
We only keep water years with at least 360 days of data.

In [65]:
flow = streamflow.load(f"{FIRST_WY - 1}-10-01", f"{LAST_WY}-09-30", gauge_ids=list(GAUGES))
flow["river"] = flow["gauge_id"].map(GAUGES)
flow["water_year"] = flow["date"].dt.year + (flow["date"].dt.month >= 10)

days = flow.groupby(["river", "water_year"]).size()
complete = days[days >= 360].reset_index()[["river", "water_year"]]
flow = flow.merge(complete).join(phase, on="water_year")
flow.groupby("river")["water_year"].agg(["min", "max", "nunique"])

,min,max,nunique
river,,,
Columbia below Priest Rapids Dam,1950,2026,77
Yakima at Kiona,1950,2026,76
Yakima at Umtanum,1950,2026,77


## When the water arrives (Average daily flow by winter ENSO phase)

Average flow on each day of the water year, for La Niña, neutral and El Niño winters. The shaded
band around each line shows the middle half of the years in that phase (25th to 75th percentile). Most of the year's
water comes with the snowmelt in May and June. A note: October to December come before
the winter that sets their phase, but El Niño and La Niña usually develop by autumn, so they are already
under way at the start of the water year.

In [66]:
COLOURS = {"La Niña": "#2471a3", "Neutral": "#909090", "El Niño": "#c0392b"}

daily = flow[~((flow["date"].dt.month == 2) & (flow["date"].dt.day == 29))].copy() # removes leap day, when it exists

# put every water year on the same calendar so the lines can be compared (using Oct 2001 to Sep 2002, but any year is the same)
daily["day"] = daily["date"].map(lambda d: d.replace(year=2001 if d.month >= 10 else 2002))

# group by gauge, ENSO phase and calendar day. Then average over all years in that phase,
# and find the range of the middle half of those years (25th to 75th percentile) for the band
shape = daily.groupby(["river", "phase", "day"], observed=True)["flow_cfs"].agg(
    mean="mean", low=lambda f: f.quantile(0.25), high=lambda f: f.quantile(0.75),
).reset_index()

from plotly.subplots import make_subplots


def faded(colour: str, opacity: float = 0.15) -> str:
    """A see-through version of a colour like "#2471a3", for the bands."""
    r, g, b = (int(colour[i:i + 2], 16) for i in (1, 3, 5))
    return f"rgba({r}, {g}, {b}, {opacity})"


rivers = list(GAUGES.values())
fig = make_subplots(rows=len(rivers), cols=1, shared_xaxes=True, subplot_titles=rivers, vertical_spacing=0.06)
for row, river in enumerate(rivers, start=1):
    for phase_name, colour in COLOURS.items():
        d = shape[(shape["river"] == river) & (shape["phase"] == phase_name)]
        # band: an invisible line at the 75th percentile, then one at the 25th filled up to it
        fig.add_scatter(x=d["day"], y=d["high"], mode="lines", line_width=0, hoverinfo="skip",
                        legendgroup=phase_name, showlegend=False, row=row, col=1)
        fig.add_scatter(x=d["day"], y=d["low"], mode="lines", line_width=0, fill="tonexty",
                        fillcolor=faded(colour), hoverinfo="skip", legendgroup=phase_name, showlegend=False,
                        row=row, col=1)
        # the average, on top of the band. One legend entry per phase (clicking it hides line and bands)
        fig.add_scatter(x=d["day"], y=d["mean"], mode="lines", line=dict(color=colour, width=1.5),
                        name=phase_name, legendgroup=phase_name, showlegend=(row == 1),
                        hovertemplate=f"{phase_name}: %{{y:,.0f}} cfs<extra></extra>", row=row, col=1)

fig.update_yaxes(title_text="Flow (cfs)", rangemode="tozero")
fig.update_xaxes(tickformat="%b")
fig.update_layout(
    title="Average daily flow by winter ENSO phase, water years 1950–2026"
          "<br><sup>Shaded: middle half of the years (25th to 75th percentile)</sup>",
    height=800, hovermode="x unified", legend=dict(orientation="h", y=-0.08, title_text="Winter"),
)
fig.show()

On the Yakima, the snowmelt in El Niño years is smaller and ends earlier, so less water is left in the river
in June, when orchards and hop yards need more and more of it. In La Niña years the high flows last
longer. On the Columbia, the difference is minimal and mostly in the height of the June peak.

Note that these time series use the mean flow over all the years in each phase and the mean can be skewed by more extreme years. The median could be used, but as it follows specific years, it is noisy as the flow goes up and down with storms and warm spells. As this analysi focuses on the differences between phases, the simple mean is appropriate. 

## The irrigation season, year by year

For each year, the average flow from April to September is calculated as a percentage of the median year. This is shown against the
winter RONI. Each dot is a single year.

Hover over a dot to see the statistics for a specific year.

In [70]:
season = (
    flow[flow["date"].dt.month.between(4, 9)] # keeps only April to September, the irrigation season
    .groupby(["river", "water_year"])["flow_cfs"].mean() # averages the daily flows into one value per gauge and year
    .reset_index()
    .join(enso, on="water_year") # adds RONI value from winter before (calculated earlier)
    .join(phase, on="water_year") # adds phase (La Niña, Neutral or El Niño; calculated earlier)
)
season["pct_of_normal"] = 100 * season["flow_cfs"] / season.groupby("river")["flow_cfs"].transform("median") # convert to % of normal

fig = px.scatter(
    season, x="roni_djf", y="pct_of_normal", color="phase", facet_col="river",
    color_discrete_map=COLOURS, category_orders={"river": list(GAUGES.values()), "phase": list(COLOURS)},
    hover_data={"water_year": True},
    trendline="ols", trendline_scope="overall", trendline_color_override="black",  # adds one linear trendline per gauge, using all years
    labels={"roni_djf": "Winter RONI (°C)", "pct_of_normal": "Apr–Sep flow (% of median)", "phase": "Winter"},
    title="Winter ENSO vs. irrigation-season flow",
)
fig.add_hline(y=100, line_dash="dot", line_color="grey", line_width=1)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

# calulates correlation between winter RONI and the flow
for col, river in enumerate(GAUGES.values(), start=1):
    d = season[season["river"] == river]
    fig.add_annotation(text=f"r = {d['roni_djf'].corr(d['pct_of_normal']):.2f}", x=0.97, y=0.97,
                       xref="x domain", yref="y domain", xanchor="right", yanchor="top", showarrow=False,
                       bgcolor="white", row=1, col=col)
    
fig.update_layout(height=450, legend=dict(orientation="h", y=-0.2))
fig.show()

## What We See

**Summary table:**

The three phase columns give the median flow of the years in that phase, as % of the gauge's normal (median) year. Basically this gives the % of normal expected for an April-September period following a winter in that phase. At Kiona, a typical La Niña year has 121% of normal water and a typical El Niño year 89%.

The correlation measures how closely flow follows winter RONI across all years. It's the same r as in the scatter chart. Negative means a warmer Pacific (El Niño) goes with less water. Around −0.3 is a real but modest link.

In [68]:
summary = season.pivot_table(index="river", columns="phase", values="pct_of_normal", aggfunc="median", observed=True)
summary = summary[list(COLOURS)].round().astype(int).add_suffix(" (% of normal)")
summary["Correlation with RONI"] = season.groupby("river").apply(lambda d: d["flow_cfs"].corr(d["roni_djf"])).round(2)
summary.loc[list(GAUGES.values())]

phase,La Niña (% of normal),Neutral (% of normal),El Niño (% of normal),Correlation with RONI
river,,,,
Yakima at Umtanum,109,99,88,-0.32
Yakima at Kiona,121,98,89,-0.30
Columbia below Priest Rapids Dam,103,108,83,-0.20


**Years per phase:**

About 25 years per phase at each gauge. The three phases are about equally common since 1950, so no phase's median rests on only a handful of years. Kiona has one year fewer because one El Niño year has too many missing days.

In [69]:
season.groupby(["river", "phase"], observed=True).size().unstack()  # number of years in each phase

phase,La Niña,Neutral,El Niño
river,,,
Columbia below Priest Rapids Dam,25,26,26
Yakima at Kiona,25,26,25
Yakima at Umtanum,25,26,26


## General Takeaways

- **El Niño winters mean less summer water in both rivers.** The median El Niño year brings about 10% less
  water than normal to the Yakima from April to September, and about 17% less to the Columbia.
- **La Niña helps the Yakima more.** At Kiona, the median La Niña year has about a fifth more water than
  normal. On the Columbia, La Niña years are not wetter than neutral years.
- **But ENSO doesn't decide a year.** The correlation is modest (−0.3 for the Yakima, −0.2 for the Columbia) and factors beyond ENSO impact streamflow. In fact, the very low irrigation season (2001) followed a La Niña winter. Precipitation type is important and 2015 was an extremely dry year even though it followed a weak El Niño winter. A significant amount of precipitation during that winter fell as rain instead of snow.
- **The same percentage means something different on each river.** Even in a dry year the Columbia carries
  far more water than the farms along it take out, and its dams can store a lot of it. The Yakima's
  reservoirs are small, and in dry summers the valley's irrigation demand is close to all the water there is.

## What it means for agriculture

NOAA's ENSO products ([discussion](https://www.cpc.ncep.noaa.gov/products/analysis_monitoring/enso_advisory/ensodisc.shtml), [outlook](https://www.cpc.ncep.noaa.gov/products/analysis_monitoring/enso/roni/outlook/), [probabilities](https://www.cpc.ncep.noaa.gov/products/analysis_monitoring/enso/roni/probabilities/), [strengths](https://www.cpc.ncep.noaa.gov/products/analysis_monitoring/enso/roni/strengths/)) are available months before the irrigation season starts, so it gives an early hint of
whether a summer is more likely to be short of water. In the Yakima Valley, that matters most for farms with
newer ("junior") water rights, which are cut back first in dry years like 2001, 2005 and 2015. ENSO changes the odds of higher/lower streamflow, but it should not be used alone to make a forecast. Snowpack measurements in late winter and spring, and the official water
supply forecasts, say much more about a specific year.

## Caveats

- Both rivers are managed: reservoirs, dams and irrigation diversions change the flow, especially at Kiona.
- Recent USGS values are provisional and can still be revised.
- A correlation over about 75 years is not proof of cause, and other things (warming winters, changes in
  water use) change flows over the same period.

*Data: U.S. Geological Survey, National Water Information System, https://doi.org/10.5066/F7P55KJN.
NOAA Climate Prediction Center, Relative Oceanic Niño Index (RONI) v6. Map tiles © OpenStreetMap contributors.*